# Tier 2 Phase 4A-Lite Forensic Pilot

Kaggle-first, Colab-compatible notebook for the 18-row long-horizon C-variant pilot: `C1_current`, `C2_dedup`, and `C3_no_latent`.

In [ ]:
import os, pathlib, zipfile, shutil
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
IS_KAGGLE = pathlib.Path('/kaggle/working').exists()
IS_COLAB = pathlib.Path('/content').exists() and not IS_KAGGLE
WORK = pathlib.Path('/kaggle/working' if IS_KAGGLE else '/content')
print({'is_kaggle': IS_KAGGLE, 'is_colab': IS_COLAB, 'work': str(WORK), 'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES')})

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!python -m pip install -q transformers==4.53.3 accelerate==1.8.1 bitsandbytes==0.46.1 "datasets>=2.20" "pandas>=2.2" "numpy>=1.26" "scikit-learn>=1.5" "sentencepiece>=0.2" "safetensors>=0.4" "psutil>=5.9" "pynvml>=11.5" "matplotlib>=3.8" "pytest>=8.0"

In [ ]:
import importlib.metadata as md, torch
for pkg in ['torch', 'transformers', 'accelerate', 'bitsandbytes']:
    print(pkg, md.version(pkg))
print('cuda_available', torch.cuda.is_available())
print('device_count', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device_name', torch.cuda.get_device_name(0))
    print('capability', torch.cuda.get_device_capability(0))

In [ ]:
zip_candidates = []
if IS_KAGGLE:
    zip_candidates.extend(pathlib.Path('/kaggle/input').rglob('latent-agent-prototype-tier2.zip'))
zip_candidates.extend(WORK.rglob('latent-agent-prototype-tier2.zip'))
if not zip_candidates:
    raise FileNotFoundError('Upload or attach latent-agent-prototype-tier2.zip before running this cell.')
zip_path = zip_candidates[0]
target = WORK / 'latent-agent-prototype'
if target.exists():
    shutil.rmtree(target)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORK)
print('extracted', zip_path, 'to', target)

In [ ]:
project = pathlib.Path('/kaggle/working/latent-agent-prototype')
if not project.exists():
    project = pathlib.Path('/content/latent-agent-prototype')
os.chdir(project)
print('cwd', os.getcwd())
!python -m pip install -q -e .

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ.setdefault('HF_TOKEN', token)
        print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as exc:
    print('Kaggle Secrets unavailable or HF_TOKEN missing:', type(exc).__name__)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ.setdefault('HF_TOKEN', token)
        print('HF_TOKEN loaded from Colab userdata')
except Exception:
    pass
print('HF_TOKEN present:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!python scripts/run_tier2_phase4.py --model Qwen/Qwen3-8B --quantization 4bit --variants C1_current,C2_dedup,C3_no_latent --families orders_kpi,sensor_quality,campaign_roi --horizons long --repeat 2 --temperature 0.2 --top-p 0.95 --base-generation-seed 17 --schedule-seed 1701 --latent-steps 4 --fallback-latent-steps 2 --latent-repair-strategy text_reset --max-new-rows 18 --output-prefix tier2_phase4_lite

In [ ]:
from pathlib import Path
checkpoint = Path('/kaggle/working/tier2_phase4_lite_checkpoint.zip') if Path('/kaggle/working').exists() else Path('/content/tier2_phase4_lite_checkpoint.zip')
result = Path('/kaggle/working/tier2_phase4_lite_results.zip') if Path('/kaggle/working').exists() else Path('/content/tier2_phase4_lite_results.zip')
print('Checkpoint zip:', checkpoint, 'exists:', checkpoint.exists())
print('Result zip:', result, 'exists:', result.exists())
!ls -lh exports reports | tail -100